# 02 · Rebuilt GCMC, the H₂–H₂ convention, and CoRE MOF 2024 checks
Reproduces report sections 5.1–5.6:
1. Pure H₂ at 77 K in an empty 30 Å box under three H₂–H₂ conventions (real H₂: 31.3 g/L at 100 bar).
2. hMOF controls and flagged labels, first setup (Feynman–Hibbs, 1.008 u).
3. The same controls with the hMOF-matched convention (2.016 u) and without the correction.
4. The eight CoRE MOF 2024 top picks.

**Runtime: about 4–6 hours on 4 cores.** **Attach:** the `mofh2_pipeline` dataset.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['02_convention_core2024']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

### 1 · Empty-box bulk H₂ density

In [ ]:
for ff, p in [('UFF_FH',100), ('UFF_FH',2), ('UFF_FHm2',100), ('UFF_noFHH2',100)]:
    !TEMP=77 bash run_box_t.sh {ff} {p} > /dev/null
!python box_density.py

### Run simulations: `02_convention_core2024` (60 runs)

In [ ]:
!bash rq_any.sh queues/02_convention_core2024.txt {MAX_HOURS}

### Analysis

In [ ]:
!python compare.py && python compare_m2.py

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG gcmc_validation.csv h2h2_convention_test.csv

In [ ]:
!python stage_io.py save stage02_outputs
# Attach /kaggle/working/stage02_outputs.tgz (via 'Save Version' output) as input to later notebooks.